# Fine-Tuned GPT-4o: Batch Evaluation & Reporting
This notebook evaluates a fine-tuned GPT-4o model on a held-out JSONL test set. It runs batched chat completions, enforces a strict JSON schema, normalizes rubric outputs, computes basic metrics (AI-segment F1; optional totals & alignment), and exports a CSV report.

## Environment Setup & Core Library Imports

This section sets up the Python environment and imports all core libraries needed for batch evaluation of the fine-tuned GPT-4o model:

- **os, json, re, time** — standard Python utilities for file paths, JSON parsing, regex, and timing requests.
- **typing** — type hints for better code clarity.
- **openai.AzureOpenAI** — Azure OpenAI client for calling your fine-tuned GPT-4o deployment.
- **statistics.mean** — used for computing average evaluation metrics (e.g., F1 score, MAE).


In [2]:
import os, json, re, time
from typing import List, Dict, Any, Tuple
from openai import AzureOpenAI
from statistics import mean

## Configuration

Set up all runtime settings and API details:

- **ENDPOINT / API_VERSION / AZURE_OPENAI_KEY** — Azure OpenAI connection info.  
- **DEPLOYMENT** — name of your fine-tuned GPT-4o model.  
- **TEST_JSONL / OUT_CSV** — paths for test data and evaluation report.  
- **TEMPERATURE / MAX_TOKENS** — model generation controls.  
- **REQ_SLEEP / TIMEOUT_S / SAMPLE_PREVIEWS** — request pacing, timeouts, and sample preview count.


In [3]:
# ========= CONFIG =========
ENDPOINT = "<ENDPOINT>"
API_VERSION = "2024-08-01-preview"  # use a portal-supported version
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")

# IMPORTANT: this must be the **deployment name** of your fine-tuned model
DEPLOYMENT = "gpt-4o-2024-08-06-ft-fcde2fe685a04119bebabe331a35da97"

TEST_JSONL = "test.jsonl"        # your holdout
OUT_CSV = "ft_eval_report.csv"
TEMPERATURE = 0                  # deterministic outputs
MAX_TOKENS = 1200
REQ_SLEEP = 0.5                  # seconds between requests
TIMEOUT_S = 120
SAMPLE_PREVIEWS = 2              # print first N predictions

## Client Setup & Helper Functions

Initialize the Azure OpenAI client and define utility functions:

- **`extract_json`** — safely parse JSON from model output (fallback if extra text is present).  
- **`to_int_score`** — convert string or float scores to integers.  
- **`parse_per_criterion_numbers`** — extract numeric scores for each rubric criterion.  
- **`pr_re_f1`** — compute precision, recall, and F1 for AI-detected text segments.


In [4]:
client = AzureOpenAI(api_version=API_VERSION, azure_endpoint=ENDPOINT, api_key=AZURE_OPENAI_KEY)

# ---------- helpers: JSON parsing ----------
def extract_json(text: str) -> Dict[str, Any]:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r'\{.*\}', text, flags=re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise ValueError("Could not parse JSON from model output.")

def to_int_score(s):
    if s is None:
        return None
    if isinstance(s, (int, float)):
        return int(s)
    if isinstance(s, str):
        # pull leading numeric like 11 in "(11/15)" or "11 points"
        m = re.search(r'(\d+)', s)
        if m:
            return int(m.group(1))
    return None

def parse_per_criterion_numbers(rubric_scores: Dict[str, Any]) -> Dict[str, int]:
    out = {}
    for crit, val in rubric_scores.items():
        if isinstance(val, dict) and "score" in val:
            out[crit] = to_int_score(val["score"])
        elif isinstance(val, str):
            m = re.search(r'\((\d+)\s*/\s*\d+\)\s*$', val.strip())
            out[crit] = int(m.group(1)) if m else None
        else:
            out[crit] = None
    return out

def pr_re_f1(true_set: set, pred_set: set) -> Tuple[float, float, float]:
    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2*prec*rec / (prec + rec) if (prec + rec) else 0.0
    return prec, rec, f1


## Reading Test Data

This part defines helper functions to load and extract the evaluation data.

`read_jsonl` reads a JSON Lines file, line by line, and converts each non-empty line into a Python dictionary. This provides the list of test examples to evaluate.

`get_ground_truth_assistant` scans each example’s messages to find the assistant’s ground truth response and parses it into a usable JSON object for comparison with the model’s predictions.


In [5]:
# ---------- read data ----------
def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def get_ground_truth_assistant(obj: Dict[str, Any]) -> Dict[str, Any]:
    msgs = obj.get("messages", [])
    for m in msgs:
        if m.get("role") == "assistant":
            return extract_json(m.get("content", ""))
    return {}

## Extracting Rubric Information

This section provides helper functions to pull rubric details from the user prompt so we know how to score and weight each criterion.

`extract_rubric_json` searches the original user message for the rubric block, parses the JSON, and returns it as a Python dictionary.  
`rubric_weight_map` then reads the criteria from that rubric and builds a simple mapping of each `criterion_id` to its assigned weight. This weight map is later used to compute total scores and to normalize outputs consistently.


In [6]:
# ---------- extract rubric (for weights) from user message ----------
def extract_rubric_json(user_content: str) -> Dict[str, Any]:
    # user content contains: "Rubric:\n{...}\n\nStudent submission:"
    m = re.search(r"Rubric:\n(\{.*?\})\n\nStudent submission:", user_content, flags=re.DOTALL)
    if not m:
        return {}
    try:
        return json.loads(m.group(1))
    except Exception:
        return {}

def rubric_weight_map(rubric: Dict[str, Any]) -> Dict[str, int]:
    weights = {}
    for c in rubric.get("criteria", []):
        cid = c.get("criterion_id")
        w = c.get("weight")
        if cid and isinstance(w, (int, float)):
            weights[cid] = int(w)
    return weights

## Building a Strict Inference Prompt

This section defines how we prepare the prompt sent to the fine-tuned GPT-4o model so the output is consistent and machine-readable.

`SCHEMA_INSTR` is a clear instruction block that tells the model exactly what JSON structure to return. It enforces:
- **No `ai_percent` field** — only `ai_segments` with paragraph indices.
- **Strict numeric scores** for each rubric criterion.
- **An integer total score**.
- **Pure JSON output** without extra commentary.

`rebuild_inference_messages` takes the original training messages and reconstructs a new user prompt for testing. It slices the stored user message to extract the assignment prompt, rubric JSON, student submission, and revision chain, then wraps these inputs with `SCHEMA_INSTR`. The result is a clean system/user message pair that guides the model to answer in the desired JSON format during evaluation.


In [7]:
# ---------- stricter inference prompt (no ai_percent; numeric scores) ----------
SCHEMA_INSTR = """You are an AI detection and grading assistant.

Return a STRICT JSON object using THIS schema:

{
  "ai_detection": { "ai_segments": [0] },
  "revision_alignment": { "alignment_percent": 0 },
  "rubric_scores": {
    "c1": { "score": 0, "comment": "" },
    "c2": { "score": 0, "comment": "" },
    "c3": { "score": 0, "comment": "" },
    "c4": { "score": 0, "comment": "" },
    "c5": { "score": 0, "comment": "" }
  },
  "total": 0
}

Rules:
- Do NOT include ai_percent. Only ai_segments (paragraph indices).
- Split the submission on double newlines (\\n\\n); index paragraphs from 0.
- All rubric 'score' fields MUST be integers (no words).
- 'total' is an integer.
- Respond with JSON only, no extra text.
"""

def rebuild_inference_messages(orig_messages: List[Dict[str, str]]) -> List[Dict[str, str]]:
    # Find the original user content so we reuse the same inputs
    user = next(m for m in orig_messages if m["role"] == "user")["content"]
    # Extract the three chunks
    def _slice(start, end, txt):
        i = txt.find(start)
        if i == -1: return ""
        j = txt.find(end, i+len(start)) if end else -1
        return (txt[i+len(start): j if j!=-1 else None]).strip()

    assignment = _slice("Assignment prompt:\n", "\n\nRubric:\n", user)
    rubric_txt = _slice("Rubric:\n", "\n\nStudent submission:\n", user)
    submission = _slice("Student submission:\n", "\n\nRevision chain info:\n", user)
    revisions = _slice("Revision chain info:\n", "", user)

    new_user = f"""{SCHEMA_INSTR}

Inputs:
Assignment prompt:
{assignment}

Rubric JSON:
{rubric_txt}

Student submission (paragraphs separated by blank lines):
{submission}

Revision chain drafts:
{revisions}
"""
    return [
        {"role": "system", "content": "You are an AI grading and detection assistant. Output valid JSON only."},
        {"role": "user", "content": new_user},
    ]


## Normalizing Model Outputs

These utilities standardize whatever the model returns into a consistent, numeric schema.  
- `WORD_TO_RATIO` maps qualitative bands (e.g., *excellent*, *good*) to a fraction of each criterion’s weight.  
- `map_word_to_score` converts such bands into integer points.  
- `normalize_rubric_scores` coerces mixed formats (strings like “(11/15) …”, qualitative bands, or dicts) into `{score: int, comment: str}` per criterion.  
- Finally, `normalize_prediction` assembles a clean result: it keeps only `ai_segments` under `ai_detection`, normalizes all rubric scores, carries over `revision_alignment` if present, and coerces `total` to an integer—yielding a uniform, machine-friendly JSON output for evaluation.


In [8]:
# ---------- normalize model outputs to numeric scores ----------
WORD_TO_RATIO = {
    # map band to fraction of the criterion's weight
    "excellent": 1.00,
    "good": 0.80,
    "average": 0.60,
    "needs_improvement": 0.40,
    "needs improvement": 0.40,
    "poor": 0.20
}

def map_word_to_score(word: str, weight: int) -> int:
    r = None
    for k, v in WORD_TO_RATIO.items():
        if k in word.lower():
            r = v
            break
    if r is None:
        return None
    return max(0, int(round(weight * r)))

def normalize_rubric_scores(pred_scores: Dict[str, Any],
                            rubric_weights: Dict[str, int]) -> Dict[str, Dict[str, Any]]:
    """
    Ensure rubric_scores are in {score:int, comment:str} form.
    If model returned words like 'excellent', convert to numeric using the rubric weight.
    If returned string with '(11/15)', extract number.
    """
    out = {}
    for cid, val in (pred_scores or {}).items():
        score_val, comment_val = None, ""

        if isinstance(val, dict):
            # already structured
            score_val = to_int_score(val.get("score"))
            comment_val = val.get("comment", "")
            # if score is None but 'score' contains words
            if score_val is None and isinstance(val.get("score"), str):
                w = rubric_weights.get(cid, 0)
                tmp = map_word_to_score(val["score"], w)
                if tmp is not None:
                    score_val = tmp

        elif isinstance(val, str):
            # try (11/15) at end
            m = re.search(r'\((\d+)\s*/\s*\d+\)\s*$', val.strip())
            if m:
                score_val = int(m.group(1))
                comment_val = val
            else:
                # try qualitative band
                w = rubric_weights.get(cid, 0)
                tmp = map_word_to_score(val, w)
                if tmp is not None:
                    score_val = tmp
                    comment_val = val
                else:
                    # last resort: pull first number anywhere
                    score_val = to_int_score(val)
                    comment_val = val
        else:
            # unknown form
            score_val = None
            comment_val = ""

        out[cid] = {"score": score_val, "comment": comment_val}
    return out

def normalize_prediction(pred: Dict[str, Any], user_msg: str) -> Dict[str, Any]:
    """Force schema: numeric rubric scores, no ai_percent, keep ai_segments, coerce total to int."""
    rubric = extract_rubric_json(user_msg)
    weights = rubric_weight_map(rubric)

    # ensure ai_detection has only ai_segments
    ai_det = pred.get("ai_detection", {}) or {}
    ai_segments = ai_det.get("ai_segments", []) or []
    if not isinstance(ai_segments, list):
        ai_segments = []

    # normalize rubric_scores
    norm_scores = normalize_rubric_scores(pred.get("rubric_scores", {}), weights)

    # coerce total
    total_int = to_int_score(pred.get("total"))

    # revision alignment (leave as-is if present)
    align_obj = pred.get("revision_alignment", {})
    # if it's missing, keep it missing

    return {
        "ai_detection": {"ai_segments": ai_segments},
        **({"revision_alignment": align_obj} if align_obj else {}),
        "rubric_scores": norm_scores,
        "total": total_int
    }


## Evaluation, Reporting & Model Invocation

This section scores each prediction and handles output.  
- `evaluate_example` compares predicted vs. ground-truth JSON: it computes precision/recall/F1 for `ai_segments`, mean absolute error (MAE) for the **total** score (when both sides are numeric), per-criterion MAE, and (when available) MAE for `revision_alignment`.  
- `write_csv` aggregates per-example metrics into a CSV for auditing and downstream analysis.  
- `call_model` sends the rebuilt system/user messages to your fine-tuned Azure OpenAI deployment, forces strict JSON via `response_format={"type":"json_object"}`, and returns the raw JSON string from the first choice.


In [9]:
# ---------- evaluation ----------
def evaluate_example(gt: Dict[str, Any], pred: Dict[str, Any]) -> Dict[str, Any]:
    gt_ai = set(gt.get("ai_detection", {}).get("ai_segments", []) or [])
    pd_ai = set(pred.get("ai_detection", {}).get("ai_segments", []) or [])
    prec, rec, f1 = pr_re_f1(gt_ai, pd_ai)

    gt_total = to_int_score(gt.get("total"))
    pd_total = to_int_score(pred.get("total"))
    total_mae = abs(pd_total - gt_total) if (gt_total is not None and pd_total is not None) else None

    gt_scores = parse_per_criterion_numbers(gt.get("rubric_scores", {}))
    pd_scores = parse_per_criterion_numbers(pred.get("rubric_scores", {}))
    percrit_abs = {}
    for k in set(gt_scores.keys()) | set(pd_scores.keys()):
        g = gt_scores.get(k)
        p = pd_scores.get(k)
        percrit_abs[k] = abs(p - g) if (g is not None and p is not None) else None

    gt_align = gt.get("revision_alignment", {}).get("alignment_percent")
    pd_align = pred.get("revision_alignment", {}).get("alignment_percent")
    align_mae = (abs(float(pd_align) - float(gt_align))
                 if (gt_align is not None and pd_align is not None)
                 else None)

    return {
        "ai_prec": prec, "ai_rec": rec, "ai_f1": f1,
        "total_mae": total_mae,
        "align_mae": align_mae,
        **{f"mae_{k}": v for k, v in percrit_abs.items()}
    }

def write_csv(rows: List[Dict[str, Any]], path: str):
    import csv
    fieldnames = set()
    for r in rows:
        fieldnames |= set(r.keys())
    fieldnames = list(sorted(fieldnames))
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow(r)

# ---------- model call ----------
def call_model(messages: List[Dict[str, str]]) -> str:
    resp = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=messages,
        temperature=TEMPERATURE,
        top_p=1.0,
        max_tokens=MAX_TOKENS,
        response_format={"type":"json_object"},  # force JSON
        timeout=TIMEOUT_S
    )
    return resp.choices[0].message.content

## Main Evaluation Loop

This is the core driver that runs the entire evaluation process.

`main()` loads all test examples from the JSONL file and iterates through each one. For every test case, it rebuilds a **strict prompt** (`rebuild_inference_messages`) and sends it to the fine-tuned GPT-4o model (`call_model`). The raw output is parsed, normalized (`normalize_prediction`), and scored against the ground truth (`evaluate_example`).

For quick inspection, the script prints the first few predictions (controlled by `SAMPLE_PREVIEWS`). It then aggregates metrics such as average F1 for AI detection, MAE for total scores, and MAE for revision alignment. All results, including per-criterion errors, are written into a CSV (`ft_eval_report.csv`) for deeper analysis.

A small sleep (`REQ_SLEEP`) is added between requests to avoid rate limits. The printed summary at the end shows the overall model performance on the test set.


In [10]:
# ---------- main ----------
def main():
    data = read_jsonl(TEST_JSONL)
    print(f"Loaded {len(data)} test examples")

    report_rows = []
    f1s, total_maes, align_maes = [], [], []
    percrit_keys = set()

    for i, obj in enumerate(data, 1):
        gt = get_ground_truth_assistant(obj)
        # rebuild a schema-enforcing prompt from original system+user
        orig_msgs = obj.get("messages", [])
        msgs = rebuild_inference_messages(orig_msgs)
        user_msg_text = msgs[-1]["content"]

        try:
            raw = call_model(msgs)
            pred_raw = extract_json(raw)
            pred = normalize_prediction(pred_raw, user_msg_text)
            eval_row = evaluate_example(gt, pred)
            status = "ok"
        except Exception as e:
            pred = {}
            eval_row = {}
            status = f"error: {e}"

        # ---- PRINT FIRST N SAMPLES FOR INSPECTION ----
        if i <= SAMPLE_PREVIEWS:
            print("\n====== SAMPLE #{} ======".format(i))
            print("Prompt preview:\n", user_msg_text[:500], "...\n")
            print("Model output JSON (normalized):\n", json.dumps(pred if pred else {"error": status}, indent=2, ensure_ascii=False))

        # aggregates
        if "ai_f1" in eval_row: f1s.append(eval_row["ai_f1"])
        if eval_row.get("total_mae") is not None: total_maes.append(eval_row["total_mae"])
        if eval_row.get("align_mae") is not None: align_maes.append(eval_row["align_mae"])
        percrit_keys |= {k for k in eval_row.keys() if k.startswith("mae_")}

        row = {
            "idx": i,
            "status": status,
            "ai_f1": eval_row.get("ai_f1"),
            "total_mae": eval_row.get("total_mae"),
            "align_mae": eval_row.get("align_mae"),
            "raw_response": json.dumps(pred, ensure_ascii=False)[:1500] if pred else status[:1500],
        }
        for k in percrit_keys:
            row[k] = eval_row.get(k)
        report_rows.append(row)

        time.sleep(REQ_SLEEP)

    write_csv(report_rows, OUT_CSV)

    def _avg(x):
        return round(mean(x), 3) if x else None

    print("\n=== Aggregate metrics on test set ===")
    print(f"AI segment F1 avg:   {_avg(f1s)}")
    print(f"Total score MAE avg: {_avg(total_maes)} (points)")
    print(f"Align % MAE avg:     {_avg(align_maes)} (percentage points)")
    print(f"CSV written to: {OUT_CSV}")

if __name__ == "__main__":
    main()


Loaded 51 test examples

====== SAMPLE #1 ======
Prompt preview:
 You are an AI detection and grading assistant.

Return a STRICT JSON object using THIS schema:

{
  "ai_detection": { "ai_segments": [0] },
  "revision_alignment": { "alignment_percent": 0 },
  "rubric_scores": {
    "c1": { "score": 0, "comment": "" },
    "c2": { "score": 0, "comment": "" },
    "c3": { "score": 0, "comment": "" },
    "c4": { "score": 0, "comment": "" },
    "c5": { "score": 0, "comment": "" }
  },
  "total": 0
}

Rules:
- Do NOT include ai_percent. Only ai_segments (paragrap ...

Model output JSON (normalized):
 {
  "ai_detection": {
    "ai_segments": [
      0,
      1,
      2,
      3,
      4,
      5
    ]
  },
  "revision_alignment": {
    "alignment_percent": 0
  },
  "rubric_scores": {
    "c1": {
      "score": 25,
      "comment": "The essay demonstrates a deep understanding of early literacy components, including phonological awareness, vocabulary, and print knowledge. It clearly explains

### Output Observations

#### Dataset
- **Loaded 51 test examples** — all samples from `test.jsonl` were read for evaluation.

#### Sample Predictions
- Shows first 2 samples to inspect model performance.
- Each sample prints:
  - **Prompt preview** — truncated input (assignment, rubric, submission).
  - **Model output JSON** — prediction in required schema:
    - `ai_detection.ai_segments` → paragraphs flagged as AI-generated.
    - `revision_alignment.alignment_percent` → similarity of revisions to final submission.
    - `rubric_scores` → numeric points + comments per criterion.
    - `total` → overall score.

#### Aggregate Metrics
- AI segment F1 avg: ~0.63 — detection of AI text is moderately good.
- Total score MAE / Align MAE: None because many ground truth samples lack numeric totals or alignment labels.
- CSV: ft_eval_report.csv contains detailed per-sample metrics.

### Takeaway

Model outputs correct JSON and is usable. Metrics for total and alignment can be added later once ground truth has full numeric data.
